In [1]:
# injected for headless run (no stdin): scripted answers, key prompts -> ''
_menu = []
_resume = []
_topic = ''
def input(prompt=''):
    p = str(prompt)
    if 'KEY' in p.upper(): ans = ''
    elif 'Pick a demo' in p: ans = _menu.pop(0) if _menu else 'q'
    elif 'topic' in p.lower(): ans = _topic
    elif p.startswith('resume>'): ans = _resume.pop(0) if _resume else '{"type": "accept"}'
    else: ans = ''
    print(f'{p.strip()[-60:]} <auto: {ans!r}>')
    return ans


# Open in Colab
<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ai-agents-the-definitive-guide/blob/main/CH03/ch03_RULER_cat_poems.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# About this Notebook

This notebook demonstrates how to **evaluate and rank language model outputs using the [OpenPipe ART](https://art.openpipe.ai/getting-started/about) (Agentic Reinforcement Tuning) framework** with the [RULER](https://art.openpipe.ai/fundamentals/ruler) reward model.

The example sets up a simple poetic-writing scenario where the model is instructed to produce **cat-themed poems**. Three different response trajectories, **high-quality**, **mediocre**, and **off-topic** generations, are manually defined to illustrate the scoring. These trajectories are then grouped and **scored automatically using RULER**, a learned evaluator that provides fine-grained quality assessments of model outputs.

The notebook illustrates:

* How to define and organize **trajectories** (`art.Trajectory`) representing model message histories and completions.
* How to evaluate multiple outputs using **`ruler_score_group`** from the `art.rewards` module.
* How to **rank responses based on reward scores**, which is essential for reinforcement-tuning workflows and preference modeling.


In [2]:
# skipped locally (Colab-only setup; the venv already has these)


In [3]:
from dotenv import load_dotenv
import os

# Load from .env if available
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")



In [4]:
import art
from art.rewards import ruler_score_group
from openai.types.chat.chat_completion import Choice
from openai.types.chat import ChatCompletionMessage

async def main():
    # Shared setup
    initial_messages = [
        {"role": "system", "content": "You are a poetic writer. Write short cat-themed poems that evoke emotion."},
        {"role": "user", "content": "Write a poem about cats observing the night sky."}
    ]

    # Three trajectories with different quality levels
    good_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Upon the roof, their eyes like stars,\n"
                        "They watch the sky from silver jars.\n"
                        "Each whisker twitches, soft delight,\n"
                        "As moons reflect their borrowed light."
                    )
                )
            )
        ],
        reward=0.0
    )

    mediocre_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Cats sit and look at stars at night,\n"
                        "They think the stars are shiny and bright."
                    )
                )
            )
        ],
        reward=0.0
    )

    off_topic_trajectory = art.Trajectory(
        messages_and_choices=[
            *initial_messages,
            Choice(
                finish_reason="stop",
                index=0,
                message=ChatCompletionMessage(
                    role="assistant",
                    content=(
                        "Dogs are great companions under the moon. "
                        "They bark at stars and wag their tails happily."
                    )
                )
            )
        ],
        reward=0.0
    )

    # Score with RULER
    group = art.TrajectoryGroup([good_trajectory, mediocre_trajectory, off_topic_trajectory])
    judged_group = await ruler_score_group(group, "openai/o3", debug=True)

    # Show ranking
    if judged_group:
        sorted_trajectories = sorted(judged_group.trajectories, key=lambda t: t.reward, reverse=True)
        for rank, traj in enumerate(sorted_trajectories, 1):
            messages = traj.messages()
            print(f"Rank {rank}: Score {traj.reward:.3f}")
            print(f"  Response: {messages[-1]['content'][:80]}...\n")

await main()


[RULER] Pretty-printed LLM choice JSON:

{
    'scores': [
        {
            'trajectory_id': '1',
            'explanation': 'Delivers an emotive, cat-themed night-sky poem with vivid imagery and good poetic form;
closely matches the goal.',
            'score': 0.9
        },
        {
            'trajectory_id': '2',
            'explanation': 'Addresses cats and the night sky but is very plain and minimally poetic; meets the goal
but with limited quality.',
            'score': 0.6
        },
        {
            'trajectory_id': '3',
            'explanation': 'Talks about dogs, not cats, so it fails the main requirement despite mentioning the 
night; almost no goal fulfillment.',
            'score': 0.05
        }
    ]
}

Rank 1: Score 0.900
  Response: Upon the roof, their eyes like stars,
They watch the sky from silver jars.
Each ...

Rank 2: Score 0.600
  Response: Cats sit and look at stars at night,
They think the stars are shiny and bright....

Rank 3: Score 0.050
  Response: Dogs are great companions under the moon. They bark at stars and wag their tails...

